# 05 — Datos procesados: mismo modelo, mejores insumos

## Pregunta que guía el notebook

Puede el mismo modelo producir una lista más útil al recibir ventas consistentes, perfiles temporales y variables con significado. No cambiaremos algoritmo, semilla, objetivo ni mes de prueba.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
def find_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/retail_customer_intelligence')]:
        if (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto. Abre la carpeta retail_customer_intelligence, no sólo el archivo .ipynb.')
ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from retail_pipeline import *
ensure_folders(ROOT)
print('Proyecto:', ROOT)

## Entrenamiento controlado

El bosque aleatorio es exactamente el mismo del baseline. Entrena con junio y julio, usa agosto para elegir el umbral y deja septiembre como prueba final. La diferencia procede del dataset, no del modelo.

In [ ]:
table = pd.read_parquet(ROOT / "data/processed/modelo_procesado.parquet")
predictions, metrics, classifier, _ = train_models(table, ROOT, "procesado")
dummy = no_purchase_baseline(predictions.compra_futura)

scorecard = pd.DataFrame({
    "regla nula: nadie compra": dummy,
    "modelo procesado": metrics["clasificacion"],
}).T.drop(columns="matriz_confusion")
display(scorecard.round(3))
print("Umbral seleccionado con agosto:", round(metrics["umbral_seleccionado_validacion"], 2))

## Revisar qué señales usa el modelo

La importancia no es causalidad. Es una herramienta para comprobar que el modelo usa variables plausibles, que los identificadores no se colaron como atajos y que las señales creadas tienen un papel real.

In [ ]:
forest = classifier.named_steps["model"]
names = classifier.named_steps["preprocess"].get_feature_names_out()
importance = pd.Series(forest.feature_importances_, index=names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
importance.head(15).sort_values().plot.barh(ax=ax, color="#2a9d8f")
ax.set(title="Variables más usadas por el bosque", xlabel="importancia; no causalidad")
plt.tight_layout()
display(importance.head(15).rename("importancia").to_frame().round(4))

## Validar el ranking, no sólo una etiqueta

Una campaña contacta una fracción limitada de la lista. Medimos la tasa de compra observada en el 10% prioritario, el lift frente a seleccionar al azar y la parte del gasto futuro que cayó dentro de ese grupo.

In [ ]:
display(pd.DataFrame([metrics["ranking_por_probabilidad"]]).round(3))
display(pd.DataFrame([metrics["ranking_por_valor_esperado"]]).round(3))

review = predictions.copy()
review["decil"] = pd.qcut(review.probabilidad_compra, 10, duplicates="drop")
calibration = review.groupby("decil", observed=True).agg(
    probabilidad_media=("probabilidad_compra", "mean"),
    compra_observada=("compra_futura", "mean"),
    clientes=("CustomerID", "size"),
    gasto_observado=("gasto_futuro", "sum"),
).reset_index()
display(calibration.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(calibration.probabilidad_media, calibration.compra_observada, marker="o", color="#2a9d8f")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="Probabilidad estimada vs compra observada", xlabel="probabilidad", ylabel="tasa de compra")
sns.barplot(data=calibration, x=calibration.index + 1, y="compra_observada", color="#457b9d", ax=axes[1])
axes[1].set(title="Compra observada por decil", xlabel="decil; 10 = mayor prioridad", ylabel="tasa")
plt.tight_layout()

## Hacer la lista revisable

Un ranking no debería ser una caja negra. Los campos de perfil permiten revisar los clientes de mayor prioridad. Aun así, el modelo estima propensión, no prueba que una promoción causará la compra.

In [ ]:
profile_columns = [
    "CustomerID", "snapshot_date", "recencia_dias", "frecuencia_facturas",
    "gasto_historico", "gasto_30d", "tendencia_gasto_30d", "tasa_devolucion",
]
profiles = table.loc[table.particion.eq("test"), profile_columns]
priority_review = predictions.merge(profiles, on=["CustomerID", "snapshot_date"])
top_count = metrics["ranking_por_probabilidad"]["clientes_contactados"]
display(priority_review.nlargest(top_count, "probabilidad_compra").head(15).round(3))

## Mensaje para clase

El modelo no conoce el futuro: recibe una historia mejor representada. Las decisiones de limpieza, recencia, ventanas y tendencia son las que se ponen a prueba en el último notebook.

Artefactos: modelo procesado, predicciones de septiembre y métricas de clasificación y ranking.